# The Sun and the Earth's Climate — Implementation
# 태양과 지구의 기후 — 구현

**Paper**: Haigh, J.D. (2007), "The Sun and the Earth's Climate", *Living Rev. Solar Phys.*, 4, 2

This notebook implements key concepts from Haigh's review of Sun-climate connections:\n이 노트북은 Haigh의 태양-기후 연결 리뷰에서 핵심 개념들을 구현합니다:

1. **Earth Energy Budget & Radiative Forcing** — 지구 에너지 수지와 복사 강제력 계산\n2. **Solar Spectral Irradiance Variability** — 태양 분광 복사의 파장별 변동 시각화\n3. **Stratospheric Ozone Photochemistry** — Chapman 반응 기반 오존 프로파일 모델\n4. **TSI-to-Temperature Response** — 기후 민감도를 이용한 에너지 균형 모델\n5. **Milankovitch Orbital Cycles** — 궤도 요소 변화와 일사량 계산\n6. **Stratosphere-Troposphere Coupling Schematic** — 성층권-대류권 결합 메커니즘 도해\n7. **IPCC Radiative Forcing Comparison** — IPCC AR4 복사 강제력 비교 재현

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: Earth Energy Budget & Radiative Forcing
## 파트 1: 지구 에너지 수지와 복사 강제력

Haigh's Figure 18 (Kiehl & Trenberth, 1997) shows the global mean energy budget.\nWe reproduce the key numbers and compute radiative forcing from TSI changes.

Haigh의 Figure 18을 기반으로 전구 평균 에너지 수지의 핵심 수치를 재현하고,\nTSI 변화로부터 복사 강제력을 계산합니다.

$$\text{RF}_\text{solar} = \frac{\Delta\text{TSI}}{4}(1-A) \approx \frac{\Delta\text{TSI}}{6}$$

$$\Delta T_g = \lambda \cdot \text{RF}, \quad \lambda \in [0.3, 1.0] \;\text{K (W m}^{-2}\text{)}^{-1}$$

In [ ]:
# === Earth Energy Budget (Kiehl & Trenberth, 1997; Haigh Fig. 18) ===

TSI = 1366.0  # W/m^2 (total solar irradiance at Earth)
A = 0.31  # planetary albedo
sigma = 5.67e-8  # Stefan-Boltzmann constant

# Global mean incoming solar radiation
S_in = TSI / 4  # geometric factor: cross-section / surface area
print(f"Incoming solar (global mean): {S_in:.0f} W/m²")

# Reflected back to space
reflected = S_in * A
print(f"Reflected (albedo = {A}):     {reflected:.0f} W/m²")

# Absorbed by Earth system
absorbed = S_in * (1 - A)
print(f"Absorbed by Earth system:     {absorbed:.0f} W/m²")

# Effective emission temperature
T_e = (absorbed / sigma) ** 0.25
print(f"Effective emission temp:      {T_e:.1f} K ({T_e - 273.15:.1f} °C)")
print(f"Actual surface temp:          ~288 K (15 °C)")
print(f"Greenhouse effect:            ~{288 - T_e:.0f} K")

print("\n--- Radiative Forcing from TSI Changes ---")

# 11-year solar cycle TSI variation
delta_TSI_11yr = 1.1  # W/m^2 (peak-to-peak)
RF_11yr = delta_TSI_11yr / 4 * (1 - A)
print(f"\n11-year cycle: ΔTSI = {delta_TSI_11yr} W/m²")
print(f"  RF = ΔTSI/4 × (1-A) = {RF_11yr:.3f} W/m²")
print(f"  Approx: ΔTSI/6     = {delta_TSI_11yr/6:.3f} W/m²")

# Since 1750 (IPCC estimate)
delta_TSI_1750 = 0.7  # W/m^2
RF_1750 = delta_TSI_1750 / 4 * (1 - A)
print(f"\nSince 1750: ΔTSI ≈ {delta_TSI_1750} W/m²")
print(f"  RF = {RF_1750:.3f} W/m² (IPCC: ~0.12 W/m²)")

# Climate sensitivity range
print("\n--- Temperature Response (ΔT = λ × RF) ---")
lambda_range = [0.3, 0.6, 1.0]  # K/(W/m^2)
for lam in lambda_range:
    dT_11yr = lam * RF_11yr
    dT_1750 = lam * RF_1750
    print(f"  λ = {lam} K/(W/m²): ΔT_11yr = {dT_11yr:.3f} K, "
          f"ΔT_since1750 = {dT_1750:.3f} K")

## Part 2: Solar Spectral Irradiance Variability
## 파트 2: 태양 분광 복사의 파장별 변동

Haigh emphasizes that while TSI varies by only ~0.08%, the fractional variation in UV\nis 10–100× larger (Figure 26, Lean 2000). We model this wavelength-dependent variability.

Haigh는 TSI가 ~0.08%만 변하지만, UV에서의 백분율 변동은 10–100배 더 크다고 강조합니다\n(Figure 26). 이 파장 의존적 변동을 모델링합니다.

In [ ]:
# === Solar Spectral Irradiance Variability (cf. Haigh Fig. 26, Lean 2000) ===

# Approximate spectral irradiance variation over the 11-year solar cycle.
# Based on Lean (2000) estimates: larger fractional changes at shorter wavelengths.

wavelength = np.array([
    120, 150, 180, 200, 220, 250, 280, 300, 320, 350,
    400, 500, 600, 800, 1000, 2000, 5000, 10000
])  # nm

# Approximate percentage change (solar max - solar min) / mean
pct_change = np.array([
    50.0, 20.0, 8.0, 5.0, 3.0, 1.5, 0.8, 0.3, 0.15, 0.1,
    0.08, 0.06, 0.05, 0.04, 0.03, 0.02, 0.01, 0.005
])  # percent

# Approximate spectral irradiance at solar minimum (W/m²/nm)
# Rough Planck-like distribution scaled to TSI
def planck_scaled(lam_nm, T=5750, scale=1.0):
    """Scaled Planck function for solar spectrum at Earth (W/m²/nm)."""
    lam_m = lam_nm * 1e-9
    h, c, k = 6.626e-34, 3e8, 1.381e-23
    B = 2 * h * c**2 / lam_m**5 / (np.exp(h * c / (lam_m * k * T)) - 1)
    # Scale: integrate Planck over all λ → σT⁴, then scale to TSI/4 at Earth
    return B * 1e-9 * scale  # W/m²/nm

lam_fine = np.linspace(100, 10000, 5000)
irrad_planck = planck_scaled(lam_fine, T=5750, scale=2.16e-5)

# Energy change (W/m²/nm) = irradiance × fractional change
irrad_at_wl = planck_scaled(wavelength, T=5750, scale=2.16e-5)
energy_change = irrad_at_wl * pct_change / 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top: Percentage change
ax1.loglog(wavelength, pct_change, "ro-", lw=2, markersize=6)
ax1.axhline(0.08, color="blue", ls="--", lw=1.5, label="Bolometric (TSI) ~0.08%")
ax1.set_ylabel("Percentage Change\n(solar max − min) / mean [%]")
ax1.set_title("Solar Spectral Irradiance Variability over 11-year Cycle\n"
              "태양 분광 복사의 11년 주기 변동 (cf. Haigh Fig. 26)")
ax1.legend(fontsize=10)
ax1.set_ylim(1e-3, 200)
ax1.set_xlim(100, 12000)

# Annotate key wavelength regions
ax1.axvspan(100, 320, alpha=0.15, color="purple", label="UV")
ax1.axvspan(320, 700, alpha=0.1, color="green", label="Visible")
ax1.axvspan(700, 12000, alpha=0.08, color="red", label="Near-IR")
ax1.legend(fontsize=9, loc="upper right")

# Bottom: Absolute energy change
ax2.loglog(wavelength, energy_change * 1e3, "bs-", lw=2, markersize=6)
ax2.set_xlabel("Wavelength [nm]")
ax2.set_ylabel("Energy Change\n(max − min) [mW m⁻² nm⁻¹]")
ax2.axvspan(100, 320, alpha=0.15, color="purple")
ax2.axvspan(320, 700, alpha=0.1, color="green")
ax2.axvspan(700, 12000, alpha=0.08, color="red")

# Key insight annotation
ax2.annotate(
    "UV: small energy but\nhuge % change →\nstratospheric impact",
    xy=(200, energy_change[3] * 1e3),
    xytext=(500, 1e-2),
    fontsize=9,
    arrowprops=dict(arrowstyle="->", color="purple"),
    color="purple",
)
ax2.annotate(
    "Visible: dominates\ntotal energy change",
    xy=(500, energy_change[11] * 1e3),
    xytext=(2000, 5e-3),
    fontsize=9,
    arrowprops=dict(arrowstyle="->", color="green"),
    color="green",
)

plt.tight_layout()
plt.show()

print("\nKey insight / 핵심 통찰:")
print("UV (<320 nm) has 10–100× larger fractional variation than visible/IR,")
print("even though visible light dominates the absolute energy change.")
print("This is why UV → stratospheric ozone pathway can amplify solar influence.")

## Part 3: Stratospheric Ozone Photochemistry (Chapman Model)
## 파트 3: 성층권 오존 광화학 (Chapman 모델)

The Chapman reactions determine stratospheric ozone. We implement a simplified\nsteady-state ozone profile model to show how UV variability affects ozone concentration.

Chapman 반응이 성층권 오존을 결정합니다. 단순화된 정상 상태 오존 프로파일 모델을 구현하여\nUV 변동이 오존 농도에 미치는 영향을 보여줍니다.

Key reactions / 핵심 반응:
- $\text{O}_2 + h\nu \rightarrow 2\text{O}$ (production, UV < 242 nm)
- $\text{O} + \text{O}_2 + \text{M} \rightarrow \text{O}_3 + \text{M}$ (ozone formation)
- $\text{O}_3 + h\nu \rightarrow \text{O}_2 + \text{O}$ (ozone photolysis, UV < 310 nm)
- $\text{O} + \text{O}_3 \rightarrow 2\text{O}_2$ (ozone destruction)

In [ ]:
# === Simplified Chapman Ozone Model ===
# Steady-state solution for ozone mixing ratio as a function of altitude.
#
# In the Chapman pure-oxygen scheme, the steady-state ozone number density is:
#   [O3] = (J2 * [O2] * k2 * [M] / (J3 * k4))^(1/2)
# where:
#   J2 = O2 photolysis rate (depends on UV flux at λ < 242 nm)
#   k2 = O + O2 + M → O3 rate (three-body, temperature dependent)
#   J3 = O3 photolysis rate (depends on UV flux at λ < 310 nm)
#   k4 = O + O3 → 2O2 rate (temperature dependent)

# Altitude grid
z = np.linspace(15, 70, 200)  # km

# Atmospheric number density profile (exponential atmosphere)
H = 7.0  # scale height in km
n_0 = 2.55e19  # number density at surface (cm^-3)
n_air = n_0 * np.exp(-z / H)  # total air number density
n_O2 = 0.21 * n_air  # O2 number density

# Temperature profile (simplified)
T = np.where(z < 20, 220 - 2 * (z - 15),
    np.where(z < 50, 220 + 1.5 * (z - 20),
             265 - 3 * (z - 50)))

# Simplified photolysis and rate coefficients
def chapman_ozone(z, n_O2, n_air, T, uv_factor=1.0):
    """Compute steady-state ozone profile from Chapman chemistry.

    Args:
        z: Altitude array (km).
        n_O2: O2 number density (cm^-3).
        n_air: Total air number density (cm^-3).
        T: Temperature (K).
        uv_factor: Multiplicative factor for UV flux (1.0 = solar min).

    Returns:
        Ozone mixing ratio (ppmv).
    """
    # J2: O2 photolysis rate — decreases with depth (UV absorbed above)
    # Parameterized as exponential attenuation with overhead O2 column
    tau_O2 = 1e-24 * np.cumsum(n_O2[::-1])[::-1] * (z[1] - z[0]) * 1e5
    J2 = 1e-11 * uv_factor * np.exp(-tau_O2)

    # J3: O3 photolysis rate — less strongly attenuated (longer λ)
    J3 = 5e-4 * uv_factor * np.exp(-z * 0.01)

    # k2: three-body reaction rate (cm^6 s^-1), increases at lower T
    k2 = 6e-34 * (300 / T) ** 2.4

    # k4: O + O3 destruction rate (cm^3 s^-1)
    k4 = 8e-12 * np.exp(-2060 / T)

    # Steady-state: [O3] = sqrt(J2 * [O2] * k2 * [M] / (J3 * k4))
    n_O3 = np.sqrt(J2 * n_O2 * k2 * n_air / (J3 * k4))

    # Convert to mixing ratio (ppmv)
    mixing_ratio = n_O3 / n_air * 1e6
    return mixing_ratio

# Solar minimum and maximum profiles
o3_min = chapman_ozone(z, n_O2, n_air, T, uv_factor=1.0)
o3_max = chapman_ozone(z, n_O2, n_air, T, uv_factor=1.03)  # 3% UV increase

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

# Left: Ozone profiles
ax1.plot(o3_min, z, "b-", lw=2, label="Solar Minimum")
ax1.plot(o3_max, z, "r-", lw=2, label="Solar Maximum (+3% UV)")
ax1.set_xlabel("Ozone Mixing Ratio [ppmv]")
ax1.set_ylabel("Altitude [km]")
ax1.set_title("Chapman Model: Ozone Profile\n오존 프로파일 (Chapman 모델)")
ax1.legend()
ax1.set_xlim(0, None)

# Right: Percentage difference
pct_diff = (o3_max - o3_min) / o3_min * 100
ax2.plot(pct_diff, z, "g-", lw=2)
ax2.axvline(0, color="k", ls="--", lw=0.5)
ax2.set_xlabel("Ozone Change [%] (solar max − min)")
ax2.set_title("Solar Cycle Ozone Response\n태양 주기 오존 반응 (cf. Haigh Fig. 29)")
ax2.set_xlim(-1, 5)

# Annotate
ax2.annotate("Peak response\n~2% in upper\nstratosphere",
             xy=(pct_diff[np.argmax(pct_diff)], z[np.argmax(pct_diff)]),
             xytext=(3, 55), fontsize=9,
             arrowprops=dict(arrowstyle="->", color="green"))

plt.tight_layout()
plt.show()

print("Chapman model shows ~1.5–2% ozone increase at 35–45 km for 3% UV increase,")
print("consistent with Haigh's discussion of ~2% observed response (Section 5.4).")

## Part 4: 1-D Energy Balance Model (EBM) with Solar Forcing
## 파트 4: 태양 강제력을 포함한 1차원 에너지 균형 모델

Following Crowley (2000) and Haigh's discussion (§6.1, Figure 32), we build a simple\n0-D energy balance model to show how TSI variations and CO₂ contribute to temperature.

Crowley (2000)와 Haigh의 논의(§6.1, Figure 32)를 따라 간단한 0차원 에너지 균형 모델을\n구축하여 TSI 변동과 CO₂가 온도에 기여하는 방식을 보여줍니다.

$$C \frac{dT}{dt} = \frac{S(t)}{4}(1-A) - \sigma T^4 + \text{GHG forcing}$$

In [ ]:
# === 0-D Energy Balance Model with Solar + GHG Forcing ===
# Linearized: ΔT(t) = λ * RF(t), with thermal inertia from a mixed-layer ocean.
#
# dT'/dt = (1/τ) * [λ * RF(t) - T'(t)]
# where τ = C * λ, C = ocean heat capacity per unit area

# Parameters
lam = 0.6  # climate sensitivity [K / (W/m²)]
C_ocean = 4.18e6 * 70  # mixed-layer ocean heat capacity [J/(m² K)]
                         # = ρ * c_p * depth ≈ 70m mixed layer
tau = C_ocean * lam  # response timescale [s]
tau_yr = tau / (365.25 * 86400)  # convert to years
print(f"Ocean thermal inertia timescale: τ = {tau_yr:.1f} years")

# Time axis: 1750–2010
years = np.arange(1750, 2011)
dt = 1.0  # year

# --- Solar forcing ---
# Simplified: 11-year cycle + long-term trend (higher in early 20th century)
solar_cycle = 0.1 * np.sin(2 * np.pi * (years - 1750) / 11)  # W/m² RF amplitude

# Long-term solar trend: increase from 1750 to ~1950, then flat
solar_trend = np.where(years < 1900, 0.05 * (years - 1750) / 150,
              np.where(years < 1950, 0.05 + 0.03 * (years - 1900) / 50,
                       0.08))
RF_solar = solar_cycle + solar_trend

# --- GHG forcing ---
# CO2: ~280 ppm in 1750, ~385 ppm in 2010
# RF_CO2 = 5.35 * ln(C/C0)
CO2_1750 = 280.0
CO2 = np.where(years < 1850, CO2_1750,
      np.where(years < 1960, CO2_1750 + 30 * (years - 1850) / 110,
               CO2_1750 + 30 + 75 * (years - 1960) / 50))
CO2 = np.minimum(CO2, 385)
RF_ghg = 5.35 * np.log(CO2 / CO2_1750)

# --- Volcanic forcing (simplified major eruptions) ---
RF_volcanic = np.zeros_like(years, dtype=float)
eruptions = {1815: -3.0, 1883: -2.5, 1902: -1.5, 1963: -1.0, 1982: -2.0, 1991: -2.5}
for yr, mag in eruptions.items():
    mask = (years >= yr) & (years < yr + 3)
    RF_volcanic[mask] += mag * np.exp(-(years[mask] - yr) / 1.0)

# --- Total forcing ---
RF_total = RF_solar + RF_ghg + RF_volcanic
RF_natural = RF_solar + RF_volcanic

# --- Integrate EBM ---
def run_ebm(RF, lam, tau_yr, dt=1.0):
    """Run 0-D EBM with given forcing time series."""
    T = np.zeros_like(RF)
    for i in range(1, len(RF)):
        dTdt = (lam * RF[i] - T[i-1]) / tau_yr
        T[i] = T[i-1] + dTdt * dt
    return T

T_total = run_ebm(RF_total, lam, tau_yr)
T_natural = run_ebm(RF_natural, lam, tau_yr)
T_solar_only = run_ebm(RF_solar, lam, tau_yr)

# --- Plot (cf. Haigh Fig. 32 / IPCC Fig. 33) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

# Top: Forcings
ax1.plot(years, RF_solar, "orange", lw=1.5, label="Solar RF")
ax1.plot(years, RF_ghg, "red", lw=2, label="GHG RF (CO₂)")
ax1.fill_between(years, RF_volcanic, 0, alpha=0.3, color="gray", label="Volcanic")
ax1.plot(years, RF_total, "k-", lw=1, alpha=0.5, label="Total RF")
ax1.set_ylabel("Radiative Forcing [W m⁻²]")
ax1.set_title("Radiative Forcings and Temperature Response (0-D EBM)\n"
              "복사 강제력과 온도 반응 (0차원 에너지 균형 모델, cf. Haigh Fig. 32)")
ax1.legend(fontsize=9, loc="upper left")
ax1.axhline(0, color="k", lw=0.5)

# Bottom: Temperature response
ax2.plot(years, T_total, "k-", lw=2, label="All forcings / 전체 강제력")
ax2.plot(years, T_natural, "b-", lw=2, label="Natural only (solar + volcanic)")
ax2.plot(years, T_solar_only, "orange", lw=1.5, ls="--",
         label="Solar only / 태양만")
ax2.set_ylabel("Temperature Anomaly [K]")
ax2.set_xlabel("Year")
ax2.legend(fontsize=9)
ax2.axhline(0, color="k", lw=0.5)

# Annotate the key divergence
ax2.annotate("Post-1950: natural forcings\ncannot explain warming\n"
             "1950년 이후: 자연 강제력만으로\n온난화 설명 불가",
             xy=(1980, T_natural[years == 1980][0]),
             xytext=(1870, 0.6), fontsize=9,
             arrowprops=dict(arrowstyle="->", color="blue"),
             color="blue")

plt.tight_layout()
plt.show()

print("\nKey result / 핵심 결과:")
print("Natural forcings (solar + volcanic) explain pre-1950 variability well,")
print("but post-1950 warming requires anthropogenic GHG forcing.")
print(f"Solar-only warming since 1750: ~{T_solar_only[-1]:.2f} K")
print(f"Total warming since 1750:      ~{T_total[-1]:.2f} K")

## Part 5: Milankovitch Orbital Cycles
## 파트 5: Milankovitch 궤도 주기

Haigh §4.1 discusses how Earth's orbital parameters (eccentricity, obliquity, precession)\nmodulate insolation on timescales of 19–413 kyr, driving glacial-interglacial transitions.

Haigh §4.1은 지구 궤도 요소(이심률, 자전축 기울기, 세차)가 19–413 kyr 시간 규모에서\n일사량을 변조하여 빙하기-간빙기 전환을 유발하는 방식을 논의합니다.

In [ ]:
# === Milankovitch Orbital Cycles (cf. Haigh Fig. 20) ===

# Time axis: -300 kyr to +100 kyr from present
t_kyr = np.linspace(-300, 100, 4000)

# Simplified orbital parameter variations (approximate sinusoidal)

# Eccentricity: periods ~100 kyr and ~413 kyr
e_mean, e_amp1, e_amp2 = 0.028, 0.019, 0.01
eccentricity = (e_mean
                + e_amp1 * np.sin(2 * np.pi * t_kyr / 100)
                + e_amp2 * np.sin(2 * np.pi * t_kyr / 413))
eccentricity = np.clip(eccentricity, 0.005, 0.058)

# Obliquity (tilt): period ~41 kyr
obliquity_mean, obliquity_amp = 23.3, 1.2  # degrees
obliquity = obliquity_mean + obliquity_amp * np.sin(2 * np.pi * t_kyr / 41)

# Precession: periods ~19 kyr and ~23 kyr (climatic precession = e * sin(ω))
precession = (eccentricity * np.sin(2 * np.pi * t_kyr / 23)
              + 0.5 * eccentricity * np.sin(2 * np.pi * t_kyr / 19))

# Summer insolation at 65°N (simplified Berger formula)
# Q_summer ∝ (1 + e*sin(ω)) / (1 - e²) × f(obliquity)
# We use a linear combination as a proxy
lat = 65  # degrees
obliq_factor = np.sin(np.radians(obliquity)) / np.sin(np.radians(obliquity_mean))
Q_65N_summer = 440 * obliq_factor + 50 * precession  # W/m² (rough scale)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Eccentricity
axes[0].plot(t_kyr, eccentricity * 100, "k-", lw=1.5)
axes[0].set_ylabel("Eccentricity\n이심률 [%]")
axes[0].set_title("Milankovitch Orbital Cycles / Milankovitch 궤도 주기\n"
                   "(cf. Haigh Fig. 20, Burroughs 1992)")

# Obliquity
axes[1].plot(t_kyr, obliquity, "b-", lw=1.5)
axes[1].set_ylabel("Obliquity\n기울기 [°]")

# Precession
axes[2].plot(t_kyr, precession, "r-", lw=1.5)
axes[2].set_ylabel("Climatic\nPrecession\n세차")
axes[2].axhline(0, color="k", lw=0.5)

# Summer insolation at 65°N
axes[3].plot(t_kyr, Q_65N_summer, "purple", lw=1.5)
axes[3].set_ylabel("Summer Q at 65°N\n[W m⁻²]")
axes[3].set_xlabel("Time [kyr from present / 현재로부터 kyr]")

# Mark present
for ax in axes:
    ax.axvline(0, color="gray", ls=":", lw=1, alpha=0.7)

axes[0].text(2, eccentricity[t_kyr == 0][0] * 100 + 0.3, "Present\n현재",
             fontsize=8, color="gray")

plt.tight_layout()
plt.show()

print("Milankovitch cycles modulate seasonal/latitudinal insolation distribution.")
print("Summer insolation at 65°N is critical for ice sheet growth/retreat.")
print(f"Eccentricity range: {eccentricity.min():.3f}–{eccentricity.max():.3f}")
print(f"Obliquity range: {obliquity.min():.1f}°–{obliquity.max():.1f}°")

## Part 6: Stratosphere-Troposphere Coupling Mechanism
## 파트 6: 성층권-대류권 결합 메커니즘 도해

This diagram illustrates the Kodera mechanism (Haigh Fig. 38) — how solar UV heating\nin the upper stratosphere propagates downward to affect tropospheric jets and the Hadley cell.

이 도해는 Kodera 메커니즘(Haigh Fig. 38)을 시각화합니다 — 상층 성층권의 태양 UV 가열이\n어떻게 하향 전파되어 대류권 제트류와 Hadley cell에 영향을 미치는지를 보여줍니다.

In [ ]:
# === Stratosphere-Troposphere Coupling Mechanism Diagram ===
# Schematic of the Kodera/Haigh mechanism (cf. Haigh Figs. 38, 42)

fig, ax = plt.subplots(1, 1, figsize=(14, 9))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect("equal")
ax.axis("off")

# Background: atmosphere layers
ax.fill_between([0, 10], 0, 2.5, color="#E8F4FD", alpha=0.8)  # Troposphere
ax.fill_between([0, 10], 2.5, 6.0, color="#FFF3E0", alpha=0.8)  # Stratosphere
ax.fill_between([0, 10], 6.0, 8.0, color="#FCE4EC", alpha=0.8)  # Upper strat/mesosphere

# Layer labels
ax.text(0.2, 1.0, "TROPOSPHERE\n대류권\n(0–15 km)", fontsize=10, fontweight="bold",
        color="#1565C0")
ax.text(0.2, 4.0, "STRATOSPHERE\n성층권\n(15–50 km)", fontsize=10, fontweight="bold",
        color="#E65100")
ax.text(0.2, 6.8, "MESOSPHERE\n중간권\n(50–80 km)", fontsize=10, fontweight="bold",
        color="#C62828")

# Tropopause line
ax.axhline(2.5, color="blue", ls="--", lw=2, alpha=0.7)
ax.text(8.5, 2.6, "Tropopause\n대류권계면", fontsize=8, color="blue")

# Step 1: Solar UV → Stratopause heating
sun = plt.Circle((8.5, 9.0), 0.6, color="gold", ec="orange", lw=2)
ax.add_patch(sun)
ax.text(8.5, 9.0, "☀", fontsize=20, ha="center", va="center")
ax.annotate("", xy=(7.0, 7.0), xytext=(8.0, 8.4),
            arrowprops=dict(arrowstyle="-|>", color="orange", lw=2.5))
ax.text(7.5, 8.0, "UV ↑\nat solar max", fontsize=9, color="orange",
        ha="center", fontweight="bold")

# Heated region
heated = mpatches.FancyBboxPatch((4.0, 5.8), 3.0, 1.5, boxstyle="round,pad=0.2",
                                  facecolor="#FF8A65", alpha=0.6, edgecolor="red", lw=2)
ax.add_patch(heated)
ax.text(5.5, 6.5, "① UV Heating\nO₃ increase\nUV 가열 + O₃ 증가",
        fontsize=9, ha="center", fontweight="bold", color="#BF360C")

# Step 2: Polar jet modification + planetary waves
ax.annotate("", xy=(5.5, 4.5), xytext=(5.5, 5.6),
            arrowprops=dict(arrowstyle="-|>", color="red", lw=2))
jet_box = mpatches.FancyBboxPatch((3.5, 3.5), 4.0, 1.2, boxstyle="round,pad=0.2",
                                   facecolor="#FFCC80", alpha=0.6, edgecolor="#F57C00", lw=2)
ax.add_patch(jet_box)
ax.text(5.5, 4.1, "② Polar Jet Modified\nPlanetary wave path altered\n극 제트 변화 → 행성파 경로 변경",
        fontsize=9, ha="center", fontweight="bold", color="#E65100")

# Planetary wave arrows (wavy)
wave_x = np.linspace(3.0, 2.5, 30)
wave_y = np.linspace(5.0, 3.5, 30) + 0.15 * np.sin(np.linspace(0, 6*np.pi, 30))
ax.plot(wave_x, wave_y, "green", lw=2)
ax.annotate("", xy=(2.5, 3.5), xytext=(2.7, 3.8),
            arrowprops=dict(arrowstyle="-|>", color="green", lw=2))
ax.text(2.2, 4.5, "Planetary\nwaves\n행성파", fontsize=8, color="green",
        ha="center", fontstyle="italic")

# Step 3: Brewer-Dobson circulation weakens
ax.annotate("", xy=(5.5, 3.0), xytext=(5.5, 3.3),
            arrowprops=dict(arrowstyle="-|>", color="purple", lw=2, ls="--"))
ax.text(8.0, 3.5, "③ Brewer-Dobson\ncirc. weakens\nB-D 순환 약화",
        fontsize=9, color="purple", fontweight="bold")

# Step 4: Tropospheric response
tropo_box = mpatches.FancyBboxPatch((3.0, 0.5), 5.0, 1.8, boxstyle="round,pad=0.2",
                                     facecolor="#BBDEFB", alpha=0.6, edgecolor="#1565C0", lw=2)
ax.add_patch(tropo_box)
ax.text(5.5, 1.4, "④ Tropospheric Response / 대류권 반응\n"
                   "• Jets weaken & shift poleward / 제트 약화 & 극 이동\n"
                   "• Hadley cell broadens / Hadley cell 확장\n"
                   "• Eddy momentum transport dominates / 에디 운동량 수송 지배",
        fontsize=9, ha="center", fontweight="bold", color="#0D47A1")

# Downward coupling arrow
ax.annotate("", xy=(5.5, 2.4), xytext=(5.5, 3.0),
            arrowprops=dict(arrowstyle="-|>", color="#1565C0", lw=3))

# Title
ax.text(5.0, 9.5, "Stratosphere–Troposphere Coupling Mechanism\n"
                   "성층권–대류권 결합 메커니즘 (Kodera/Haigh)",
        fontsize=13, ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## Part 7: IPCC AR4 Radiative Forcing Comparison
## 파트 7: IPCC AR4 복사 강제력 비교 재현

Reproducing the key comparison from Haigh's Figure 19 (IPCC 2007) showing radiative\nforcings from various agents since 1750 — the definitive context for solar contributions.

Haigh의 Figure 19 (IPCC 2007)의 핵심 비교를 재현합니다 — 1750년 이후 다양한 요인의\n복사 강제력을 보여주어 태양 기여의 정량적 맥락을 확립합니다.

In [ ]:
# === IPCC AR4 Radiative Forcing Comparison (cf. Haigh Fig. 19) ===

# Data from IPCC AR4 (2007), Table 2.12
components = [
    "CO₂",
    "CH₄",
    "N₂O",
    "Halocarbons",
    "Tropospheric O₃",
    "Stratospheric O₃",
    "Stratospheric H₂O\n(from CH₄)",
    "Surface albedo\n(land use)",
    "Surface albedo\n(black carbon on snow)",
    "Aerosol\n(direct effect)",
    "Aerosol\n(cloud albedo)",
    "Linear contrails",
    "Solar irradiance",
]

# Best estimate RF (W/m²)
rf_values = [1.66, 0.48, 0.16, 0.34, 0.35, -0.05, 0.07, -0.2, 0.1,
             -0.5, -0.7, 0.01, 0.12]

# Uncertainty ranges (low, high)
rf_low = [1.49, 0.43, 0.14, 0.31, 0.25, -0.15, 0.02, -0.4, 0.0,
          -0.9, -1.8, 0.003, 0.06]
rf_high = [1.83, 0.53, 0.18, 0.37, 0.65, 0.05, 0.12, 0.0, 0.2,
           -0.1, -0.3, 0.03, 0.30]

# Colors
colors = []
for v in rf_values:
    if v > 0.3:
        colors.append("#D32F2F")  # strong positive
    elif v > 0:
        colors.append("#FF8A65")  # weak positive
    elif v > -0.3:
        colors.append("#64B5F6")  # weak negative
    else:
        colors.append("#1565C0")  # strong negative

# Highlight solar
colors[-1] = "#FF9800"  # orange for solar

fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(components))
bars = ax.barh(y_pos, rf_values, color=colors, edgecolor="k", lw=0.5, height=0.7)

# Error bars
for i, (low, high, val) in enumerate(zip(rf_low, rf_high, rf_values)):
    ax.plot([low, high], [i, i], "k-", lw=1.5)
    ax.plot([low, low], [i-0.15, i+0.15], "k-", lw=1.5)
    ax.plot([high, high], [i-0.15, i+0.15], "k-", lw=1.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(components, fontsize=9)
ax.set_xlabel("Radiative Forcing [W m⁻²]", fontsize=12)
ax.set_title("Radiative Forcing of Climate (1750–2005)\n"
             "기후의 복사 강제력 (1750–2005)\n"
             "(cf. Haigh Fig. 19, IPCC AR4)", fontsize=13)
ax.axvline(0, color="k", lw=1)
ax.set_xlim(-2.5, 2.5)

# Highlight solar bar
bars[-1].set_edgecolor("orange")
bars[-1].set_linewidth(3)
ax.annotate(f"Solar: {rf_values[-1]} W/m²\n= CO₂의 1/{rf_values[0]/rf_values[-1]:.0f}",
            xy=(rf_values[-1], len(components) - 1),
            xytext=(1.5, len(components) - 1),
            fontsize=10, fontweight="bold", color="#FF6F00",
            arrowprops=dict(arrowstyle="->", color="orange", lw=2))

# Category labels
ax.text(-2.4, 10.5, "Human\nActivities\n인위적",
        fontsize=9, fontweight="bold", color="gray", va="center")
ax.axhline(11.5, color="gray", ls=":", lw=1)
ax.text(-2.4, 12, "Natural\n자연적",
        fontsize=9, fontweight="bold", color="gray", va="center")

plt.tight_layout()
plt.show()

print("\n=== Summary / 요약 ===")
print(f"Total anthropogenic RF: ~{sum(rf_values[:-1]):.2f} W/m²")
print(f"Solar RF:                {rf_values[-1]} W/m²")
print(f"Solar / CO₂ ratio:       1/{rf_values[0]/rf_values[-1]:.0f}")
print(f"\nSolar forcing is real but small — ~{rf_values[-1]/sum(v for v in rf_values[:-1] if v > 0)*100:.0f}% "
      f"of total positive anthropogenic forcing.")

## Summary / 요약

| Part | Topic / 주제 | Key Result / 핵심 결과 |
|------|------------|---------------------|
| 1 | Energy Budget & RF / 에너지 수지 | TSI → RF 변환: RF ≈ ΔTSI/6. 11년 주기 RF ≈ 0.19 W/m², ΔT < 0.2 K |
| 2 | Spectral Variability / 분광 변동 | UV 변동은 TSI 대비 10–100× 더 큼 → 성층권 영향의 핵심 |
| 3 | Ozone Photochemistry / 오존 광화학 | Chapman 모델: 3% UV 증가 시 상층 성층권 O₃ ~2% 증가 |
| 4 | EBM with Solar Forcing / 에너지 균형 모델 | 1950년 이후 온난화는 태양만으로 설명 불가 — GHG 필수 |
| 5 | Milankovitch Cycles / 궤도 주기 | e, θ, p 변동이 10⁴–10⁵년 규모 기후 전환 유발 |
| 6 | Strat-Trop Coupling / 성-대 결합 | UV → O₃ → 극 제트 → 행성파 → Hadley cell 변화 |
| 7 | IPCC RF Comparison / RF 비교 | 태양 RF (0.12) = CO₂ (1.66)의 1/14 |

**Next paper in LRSP series / 다음 논문**: Check `reading_list.md` for #12